# 2 · Iterative Gaussian approximation

Independent representative LIF trials replace the recurrent graph. Clean implementation informed by HOLYKALE.ipynb, with compiled Brian2 dynamics and consistent spectral normalization.

## Shared parameters
The first generation receives independent Poisson recurrent streams at `initial_rate` (71 Hz by default; try 15 Hz as in the Colab). Subsequent generations use Gaussian input. No network output is used to initialize or fit the approximation.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sc_cortex.config import Config
from sc_cortex.results import save_result, load_result

c = Config.load('config.json')
print(c)
print('Run ID:', c.fingerprint)


## Self-consistent input
For equal E/I output statistics, recurrent mean voltage increment per second is $J(C_E-gC_I)r$ and its fluctuation spectrum is $J^2(C_E+g^2C_I)S(f)$. Gaussian increments are synthesized with SciPy FFTs, then passed to independent Brian2 LIF trials. Constant drive adds no external noise; optional external Poisson input is applied separately. Common delays drop out of single-input auto-spectra when cross-correlations are neglected.

The finite record uses the first positive spectral bin to approximate unmeasured near-zero power. Default `mixing=1` is a direct update; reducing it is optional damping, not a claim of reproducing the 2018 stabilization procedure.

In [ ]:
from sc_cortex.models import run_iteration
approximation = run_iteration(c)
path = save_result('approximation', c, approximation)
print('Saved:', path)
print('Converged:', approximation['converged'])
if not approximation['converged']:
    print('Iteration limit reached: inspect residuals; do not label this a converged solution.')

## Inspect iterations
Convergence requires both relative rate and raw-spectrum residuals below tolerance for three consecutive generations. Monte Carlo error can prevent this criterion being met; increase trials/record length and check several seeds.

In [ ]:
h = approximation['history']
fig, ax = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
ax[0].plot(h[:,0], h[:,1], '--', label='Input rate')
ax[0].plot(h[:,0], h[:,2], 'o-', color='#dc8b45', label='Output rate')
ax[0].set(xlabel='Iteration', ylabel='Rate (Hz)'); ax[0].legend()
ax[1].semilogy(h[:,0], np.maximum(h[:,3],1e-12), label='Rate')
ax[1].semilogy(h[:,0], np.maximum(h[:,4],1e-12), label='Spectrum')
ax[1].axhline(c.tolerance, color='gray', ls=':')
ax[1].set(xlabel='Iteration', ylabel='Relative residual'); ax[1].legend()
f = approximation['freq']; keep = (f>0)&(f<=300)
for k in sorted(set([0, len(h)//2, len(h)-1])):
    ax[2].semilogy(f[keep], approximation['spectra'][k,keep], label=f'Iteration {k+1}')
ax[2].set(xlabel='Frequency (Hz)', ylabel='Single-neuron PSD (Hz)'); ax[2].legend()
plt.show()